# LABORATÓRIO 09 — Arquitetura RAG Avançada
## HNSW + HyDE + Cross-Encoders


**Objetivo:** Construir um pipeline RAG de nível de produção para busca semântica em manuais médicos técnicos, superando a limitação da similaridade de cosseno pura quando o usuário faz perguntas em linguagem coloquial.

**Problema:** A query *"dor de cabeça latejante e luz incomodando"* está geometricamente longe de *"cefaleia pulsátil com fotofobia"* no espaço vetorial, mesmo sendo semanticamente equivalente.

**Solução:**
```
Query coloquial → [HyDE] → Doc. Hipotético → [HNSW] → Top-10 → [Cross-Encoder] → Top-3
```

| Passo | Técnica | Função |
|:---:|---|---|
| 1 | **HNSW** (FAISS) | Indexar o corpus como grafo hierárquico |
| 2 | **HyDE** | Transformar query coloquial em jargão técnico |
| 3 | **Bi-Encoder** | Busca rápida → Top-10 candidatos |
| 4 | **Cross-Encoder** | Re-ranking preciso → Top-3 finais |

---
##  Instalação das Dependências

>  **Execute esta célula primeiro e aguarde a instalação terminar.**

In [1]:
!pip install -q faiss-cpu sentence-transformers
print(" Dependências instaladas com sucesso!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 32.5 MB/s eta 0:00:00
 Dependências instaladas com sucesso!


## Imports e Configurações Globais

In [2]:
import numpy as np
from typing import List, Dict
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

# Modelos
EMBEDDING_MODEL     = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Hiperparâmetros do funil de recuperação
TOP_K_RETRIEVE = 10   # documentos recuperados pelo Bi-Encoder
TOP_K_RERANK   = 3    # documentos finais após Cross-Encoder

print(" Imports OK")
print(f"   Bi-Encoder  : {EMBEDDING_MODEL}")
print(f"   Cross-Encoder: {CROSS_ENCODER_MODEL}")
print(f"   Funil       : Top-{TOP_K_RETRIEVE} → Top-{TOP_K_RERANK}")

 Imports OK
   Bi-Encoder  : sentence-transformers/all-MiniLM-L6-v2
   Cross-Encoder: cross-encoder/ms-marco-MiniLM-L-6-v2
   Funil       : Top-10 → Top-3


## PASSO 0 — Corpus Simulado

22 fragmentos de manuais médicos técnicos cobrindo neurologia, farmacologia, emergências e protocolos clínicos.

In [3]:
MEDICAL_CORPUS: List[Dict] = [
    {
        "id": 0,
        "title": "Manual de Neurologia Clínica – Cap. 3",
        "text": (
            "Cefaleia pulsátil acompanhada de fotofobia e fonofobia é o quadro clássico da "
            "enxaqueca (migrânea) sem aura. A dor localiza-se unilateralmente na região "
            "frontotemporal e pode durar entre 4 e 72 horas. O tratamento agudo de primeira "
            "linha inclui analgésicos AINEs e triptanos."
        ),
    },
    {
        "id": 1,
        "title": "Protocolo de Urgências Neurológicas",
        "text": (
            "Cefaleia em trovoada (thunderclap headache) de início súbito e intensidade máxima "
            "em menos de 60 segundos deve ser considerada hemorragia subaracnoidea até prova em "
            "contrário. Solicitar TC de crânio sem contraste imediatamente."
        ),
    },
    {
        "id": 2,
        "title": "Manual de Oftalmologia – Sintomas Associados",
        "text": (
            "Fotofobia, definida como hipersensibilidade dolorosa à luz, pode ser sintoma de "
            "meningite, cefaleia em salvas, uveíte anterior ou enxaqueca. A avaliação diferencial "
            "inclui inspeção do reflexo pupilar e avaliação de rigidez de nuca."
        ),
    },
    {
        "id": 3,
        "title": "Farmacologia Clínica – Analgésicos",
        "text": (
            "Os triptanos (sumatriptana, rizatriptana) agem como agonistas seletivos dos receptores "
            "5-HT1B/1D, promovendo vasoconstrição das artérias intracranianas e inibição da "
            "liberação de neuropeptídeos pró-inflamatórios. Contraindicados em doença cardiovascular isquêmica."
        ),
    },
    {
        "id": 4,
        "title": "Semiologia Médica – Avaliação da Dor",
        "text": (
            "A escala visual analógica (EVA) quantifica a intensidade da dor de 0 a 10. Dores "
            "pulsáteis ou latejantes correlacionam-se frequentemente com mecanismo vascular, "
            "enquanto dores constritivas ou em pressão sugerem cefaleia tensional."
        ),
    },
    {
        "id": 5,
        "title": "Manual de Neurologia – Cefaleia Tensional",
        "text": (
            "A cefaleia tensional episódica caracteriza-se por pressão bilateral, não pulsátil, "
            "de intensidade leve a moderada, sem náuseas e sem agravamento pela atividade física "
            "rotineira. É o tipo de cefaleia mais prevalente na população geral."
        ),
    },
    {
        "id": 6,
        "title": "Protocolo Clínico – Meningite Bacteriana",
        "text": (
            "A tríade clássica da meningite bacteriana aguda inclui febre alta, rigidez de nuca "
            "(meningismo) e alteração do nível de consciência. Cefaleia intensa, fotofobia e "
            "vômitos em jato completam o quadro. Punção lombar é obrigatória na ausência de "
            "contraindicações neurológicas."
        ),
    },
    {
        "id": 7,
        "title": "Diretriz de Hipertensão Arterial Sistêmica",
        "text": (
            "A crise hipertensiva (PA > 180/120 mmHg) pode manifestar-se com cefaleia occipital "
            "pulsátil, epistaxe, visão turva e dispneia. A emergência hipertensiva ocorre quando "
            "há lesão de órgão-alvo: encefalopatia, infarto agudo ou dissecção aórtica."
        ),
    },
    {
        "id": 8,
        "title": "Manual de Otorrinolaringologia – Sinusite",
        "text": (
            "A sinusite maxilar aguda bacteriana causa dor facial de caráter pressivo na região "
            "malar e frontal, agravada pela posição ortostática. Cefaleia frontal intensa ao "
            "inclinar a cabeça para frente é sinal característico de sinusite frontal."
        ),
    },
    {
        "id": 9,
        "title": "Neurologia – Cefaleia em Salvas (Cluster Headache)",
        "text": (
            "A cefaleia em salvas é uma forma de cefaleia trigeminoautonômica caracterizada por dor "
            "unilateral periorbital de intensidade excruciante, com duração de 15 a 180 minutos. "
            "Sintomas autonômicos ipsilaterais: lacrimejamento, injeção conjuntival, ptose e rinorreia."
        ),
    },
    {
        "id": 10,
        "title": "Manual de Pediatria – Cefaleia na Infância",
        "text": (
            "Crianças com cefaleia recorrente devem ser avaliadas para enxaqueca pediátrica, que "
            "frequentemente se apresenta de forma bilateral, com episódios mais curtos (1-72 horas). "
            "Dor abdominal e cinetose são comorbidades comuns nesta faixa etária."
        ),
    },
    {
        "id": 11,
        "title": "Tratado de Medicina Interna – Tireoidopatias",
        "text": (
            "O hipotireoidismo pode causar cefaleia crônica difusa, fadiga intensa, ganho de peso, "
            "bradicardia e intolerância ao frio. O diagnóstico baseia-se na dosagem sérica de TSH "
            "elevado e T4 livre reduzido. Tratamento com levotiroxina sódica."
        ),
    },
    {
        "id": 12,
        "title": "Protocolo de Neuroimagem – Indicações de TC",
        "text": (
            "Indicações absolutas de TC de crânio em cefaleia: início súbito ('pior dor da vida'), "
            "cefaleia progressiva sem melhora, associação com febre e rigidez de nuca, déficit "
            "neurológico focal, papiledema ao fundo de olho ou pós-trauma craniano."
        ),
    },
    {
        "id": 13,
        "title": "Manual de Anestesiologia – Dor Neuropática",
        "text": (
            "A dor neuropática caracteriza-se por sensação de queimação, choque elétrico ou "
            "formigamento ao longo de um dermátomo. A neuralgia do trigêmeo provoca dores faciais "
            "lancinantes unilaterais, geralmente desencadeadas por estímulos táteis leves (allodynia)."
        ),
    },
    {
        "id": 14,
        "title": "Guia de Psiquiatria Clínica – Transtornos de Ansiedade",
        "text": (
            "O transtorno de ansiedade generalizada (TAG) frequentemente cursa com cefaleia tensional "
            "crônica, insônia, tensão muscular cervical, irritabilidade e dificuldade de concentração. "
            "O tratamento inclui TCC e inibidores seletivos de recaptação de serotonina (ISRS)."
        ),
    },
    {
        "id": 15,
        "title": "Semiologia – Exame Neurológico Básico",
        "text": (
            "O exame neurológico sumário inclui: avaliação do nível de consciência pela Escala de "
            "Glasgow, pares cranianos (II a XII), força motora dos quatro membros, reflexos "
            "tendinosos profundos, sensibilidade e coordenação cerebelar (diadococinesia, prova índex-nariz)."
        ),
    },
    {
        "id": 16,
        "title": "Manual de Geriatria – Cefaleia no Idoso",
        "text": (
            "Arterite de células gigantes (arterite temporal) deve ser suspeitada em pacientes acima "
            "de 50 anos com cefaleia temporal nova, claudicação de mandíbula e VHS elevada. "
            "Risco de cegueira por oclusão da artéria oftálmica. Iniciar corticoide empiricamente."
        ),
    },
    {
        "id": 17,
        "title": "Farmacologia – Analgesia Escalonada (OMS)",
        "text": (
            "A escada analgésica da OMS propõe: degrau 1 (dipirona, paracetamol, AINEs para dor leve), "
            "degrau 2 (tramadol para dor moderada) e degrau 3 (morfina para dor intensa). "
            "Adjuvantes como antidepressivos tricíclicos podem ser acrescentados em qualquer degrau."
        ),
    },
    {
        "id": 18,
        "title": "Protocolo de Pós-Operatório Neurocirúrgico",
        "text": (
            "Cefaleia pós-raquianestesia (cefaleia postural) é consequência do vazamento de LCR "
            "pelo orifício de punção. Caracteriza-se por dor intensa ao sentar, aliviada em decúbito "
            "dorsal. Tratamento: repouso, hidratação, cafeína oral ou blood patch epidural em casos refratários."
        ),
    },
    {
        "id": 19,
        "title": "Diretriz de Cefaleia Crônica Diária",
        "text": (
            "Cefaleia crônica diária é definida como dor de cabeça presente em mais de 15 dias por "
            "mês durante pelo menos 3 meses. Causas: enxaqueca crônica, cefaleia por uso excessivo "
            "de medicamentos (rebote analgésico), cefaleia tensional crônica e hemicrania contínua."
        ),
    },
    {
        "id": 20,
        "title": "Manual de Emergências – Hipertensão Intracraniana",
        "text": (
            "Síndrome de hipertensão intracraniana: cefaleia progressiva matinal, vômitos em jato sem "
            "náusea prévia, papiledema bilateral e comprometimento progressivo da consciência. "
            "Causas: tumor cerebral, hematoma subdural, abscesso encefálico e hidrocefalia obstrutiva."
        ),
    },
    {
        "id": 21,
        "title": "Protocolo de AVC – Janela Terapêutica",
        "text": (
            "O AVC isquêmico agudo pode ser tratado com trombólise intravenosa (alteplase 0,9 mg/kg) "
            "dentro de uma janela terapêutica de até 4,5 horas do início dos sintomas. "
            "A trombectomia mecânica é indicada para oclusões de grandes vasos até 24 horas."
        ),
    },
]

print(f" Corpus carregado: {len(MEDICAL_CORPUS)} fragmentos de manuais médicos.")
print("\nPrimeiros 3 documentos:")
for doc in MEDICAL_CORPUS[:3]:
    print(f"  [{doc['id']}] {doc['title']}")

 Corpus carregado: 22 fragmentos de manuais médicos.

Primeiros 3 documentos:
  [0] Manual de Neurologia Clínica – Cap. 3
  [1] Protocolo de Urgências Neurológicas
  [2] Manual de Oftalmologia – Sintomas Associados


## PASSO 1 — Construção do Índice HNSW com FAISS

### Por que HNSW e não KNN exato?

O **KNN exato** compara a query contra **todos os N vetores** → complexidade `O(N × d)`. Para 1 milhão de documentos, isso significa ~500 ms por query.

O **HNSW** (Hierarchical Navigable Small World) constrói um **grafo multicamada** onde cada camada é um "Small World" — um grafo esparso com atalhos de longa distância. A busca percorre do topo (poucas conexões, visão macro) até a base (muitas conexões, precisão fina), com complexidade `O(log N)` → **< 1 ms** por query.

### Hiperparâmetros e impacto na RAM

| Parâmetro | Papel | Valor usado |
|---|---|:---:|
| **`M`** | Conexões bidirecionais por nó. Maior M = mais recall, mais RAM | `32` |
| **`ef_construction`** | Tamanho da fila durante a indexação. Maior = grafo melhor, indexação mais lenta | `200` |
| **`ef_search`** | Tamanho da fila durante a busca. Ajustável sem re-indexar | `50` |

**Fórmula de RAM aproximada:**
```
RAM ≈ N × d × 4 bytes (vetores) + N × M × 2 × 8 bytes (ponteiros do grafo)
```
O HNSW usa ~15–20% mais RAM que armazenar apenas os vetores, mas elimina o custo quadrático de busca.

In [4]:
# Carregar o Bi-Encoder
print(" Carregando modelo de embedding (Bi-Encoder)...")
bi_encoder = SentenceTransformer(EMBEDDING_MODEL)
print(f" Modelo '{EMBEDDING_MODEL}' carregado.")

# Gerar embeddings para todos os documentos
print("\n Gerando embeddings do corpus...")
texts = [doc["text"] for doc in MEDICAL_CORPUS]
embeddings = bi_encoder.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
).astype(np.float32)

# Normalizar os vetores (L2) para que o Inner Product seja equivalente à
# Similaridade de Cosseno — obrigatório ao usar IndexHNSWFlat
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
print(f"Embeddings gerados. Shape: {embeddings.shape} | Dimensão: {dim}")

# Construir índice HNSW
M               = 32    # conexões por nó
ef_construction = 200   # qualidade do grafo na indexação
ef_search       = 50    # fila dinâmica na busca

index = faiss.IndexHNSWFlat(dim, M)
index.hnsw.efConstruction = ef_construction
index.hnsw.efSearch       = ef_search

index.add(embeddings)

print(f"\n{'='*55}")
print(f"  ÍNDICE HNSW CRIADO COM SUCESSO")
print(f"{'='*55}")
print(f"  Vetores indexados : {index.ntotal}")
print(f"  Dimensão          : {dim}")
print(f"  M (conexões)      : {M}")
print(f"  ef_construction   : {ef_construction}")
print(f"  ef_search         : {ef_search}")

# Estimativa de RAM
ram_vetores   = index.ntotal * dim * 4
ram_ponteiros = index.ntotal * M * 2 * 8
ram_total     = ram_vetores + ram_ponteiros
print(f"\n  RAM estimada (vetores)   : {ram_vetores:,} bytes")
print(f"  RAM estimada (grafo)     : {ram_ponteiros:,} bytes")
print(f"  RAM total estimada       : {ram_total:,} bytes ({ram_total/1024:.1f} KB)")

 Carregando modelo de embedding (Bi-Encoder)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Modelo 'sentence-transformers/all-MiniLM-L6-v2' carregado.

 Gerando embeddings do corpus...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings gerados. Shape: (22, 384) | Dimensão: 384

  ÍNDICE HNSW CRIADO COM SUCESSO
  Vetores indexados : 22
  Dimensão          : 384
  M (conexões)      : 32
  ef_construction   : 200
  ef_search         : 50

  RAM estimada (vetores)   : 33,792 bytes
  RAM estimada (grafo)     : 11,264 bytes
  RAM total estimada       : 45,056 bytes (44.0 KB)


##  PASSO 2 — HyDE: Hypothetical Document Embeddings

### O problema geométrico

```
Espaço vetorial (simplificado em 2D):

  "cefaleia pulsátil" ●  ● "fotofobia"       ← documentos técnicos
                    ↑
           grande distância
                    ↓
  "dor de cabeça latejante e luz incomodando" ●  ← query do usuário
```

### A solução HyDE
O LLM gera uma resposta técnica **hipotética** (pode ser factualmente imprecisa). O que importa é que o vetor desse documento falso **está no mesmo espaço semântico** dos documentos reais.

> **Em produção:** substituir a função abaixo por uma chamada real à API de um LLM (OpenAI, Anthropic, etc.) com o prompt: *"Gere uma resposta técnica médica para: {query}"*

In [5]:
def generate_hypothetical_document(query: str) -> str:
    """
    Simula a chamada ao LLM para geração do Documento Hipotético (HyDE).

    Em produção, substituir pelo SDK da OpenAI / Anthropic:
        client.messages.create(
            model="claude-sonnet-4-20250514",
            messages=[{"role": "user", "content": f"Gere uma resposta técnica médica para: {query}"}]
        )
    """
    hyde_map = {
        "dor de cabeça latejante e luz incomodando": (
            "Cefaleia pulsátil unilateral de moderada a alta intensidade, acompanhada de fotofobia "
            "e fonofobia, com duração de 4 a 72 horas, agravada pela atividade física rotineira. "
            "Quadro consistente com enxaqueca (migrânea) sem aura conforme critérios ICHD-3. "
            "Tratamento agudo de primeira linha: triptanos (sumatriptana) e AINEs."
        ),
        "pressão alta dor na cabeça": (
            "Crise hipertensiva com cefaleia occipital pulsátil, PA sistólica acima de 180 mmHg. "
            "Investigar lesão de órgão-alvo: encefalopatia hipertensiva, AVC hemorrágico, "
            "insuficiência renal aguda. Anti-hipertensivos intravenosos em urgência."
        ),
        "dor de cabeça todo dia ha meses": (
            "Cefaleia crônica diária presente em mais de 15 dias por mês durante período superior "
            "a 3 meses. Investigar cefaleia por uso excessivo de analgésicos (rebote), enxaqueca "
            "crônica ou cefaleia tensional crônica. Suspender analgésicos de curta ação."
        ),
        "tontura e vomito com ouvido tampado": (
            "Síndrome vestibular periférica com vertigem rotatória, náuseas, vômitos e plenitude "
            "auricular. Provável labirintite viral ou doença de Ménière. "
            "Audiometria e vestibulolabirinto indicados. Betaistina e supressores vestibulares."
        ),
    }

    return hyde_map.get(
        query.lower().strip(),
        (
            f"Quadro clínico compatível com: {query}. Avaliação de cefaleia primária versus "
            "secundária conforme critérios ICHD-3. Exame neurológico completo indicado. "
            "Neuroimagem conforme sinais de alerta clínicos."
        ),
    )


# Definir a query do usuário
USER_QUERY = "dor de cabeça latejante e luz incomodando"

hypothetical_doc = generate_hypothetical_document(USER_QUERY)

print("HyDE — Transformação da Query")
print(f"\n  Query coloquial do usuário:")
print(f"     '{USER_QUERY}'")
print(f"\n  Documento hipotético gerado pelo LLM (jargão técnico):")
print(f"     '{hypothetical_doc}'")

HyDE — Transformação da Query

  Query coloquial do usuário:
     'dor de cabeça latejante e luz incomodando'

  Documento hipotético gerado pelo LLM (jargão técnico):
     'Cefaleia pulsátil unilateral de moderada a alta intensidade, acompanhada de fotofobia e fonofobia, com duração de 4 a 72 horas, agravada pela atividade física rotineira. Quadro consistente com enxaqueca (migrânea) sem aura conforme critérios ICHD-3. Tratamento agudo de primeira linha: triptanos (sumatriptana) e AINEs.'


## PASSO 3 — Busca Rápida via Bi-Encoder + HNSW (Top-10)

O vetor do **documento hipotético** (não da query original!) é usado para buscar os documentos mais próximos no índice HNSW.

Este é o **"funil largo"** — priorizamos recall alto, podendo trazer alguns documentos menos relevantes que serão filtrados no próximo passo.

In [6]:
# Vetorizar o documento hipotético
query_vec = bi_encoder.encode(
    [hypothetical_doc], convert_to_numpy=True
).astype(np.float32)
faiss.normalize_L2(query_vec)  # normalizar da mesma forma que o corpus

# Buscar Top-K no índice HNSW
distances, indices_found = index.search(query_vec, TOP_K_RETRIEVE)

# Montar lista de candidatos
candidates = []
for rank, (dist, idx) in enumerate(zip(distances[0], indices_found[0]), 1):
    doc = MEDICAL_CORPUS[idx].copy()
    doc["bi_encoder_score"] = float(dist)
    doc["bi_encoder_rank"]  = rank
    candidates.append(doc)

# Imprimir resultado
print(f" Top-{TOP_K_RETRIEVE} documentos recuperados pelo Bi-Encoder + HNSW")
print(f"   (vetor do documento hipotético → índice HNSW)")
print()
print(f"  {'Rank':<5} {'Score':>8}   Título")
print("  " + "─" * 65)
for doc in candidates:
    print(f"  {doc['bi_encoder_rank']:<5} {doc['bi_encoder_score']:>8.4f}   {doc['title']}")

 Top-10 documentos recuperados pelo Bi-Encoder + HNSW
   (vetor do documento hipotético → índice HNSW)

  Rank     Score   Título
  ─────────────────────────────────────────────────────────────────
  1       0.4550   Manual de Neurologia Clínica – Cap. 3
  2       0.7923   Manual de Oftalmologia – Sintomas Associados
  3       0.9293   Manual de Neurologia – Cefaleia Tensional
  4       1.0782   Protocolo de Urgências Neurológicas
  5       1.1321   Farmacologia Clínica – Analgésicos
  6       1.1326   Neurologia – Cefaleia em Salvas (Cluster Headache)
  7       1.1455   Tratado de Medicina Interna – Tireoidopatias
  8       1.1512   Protocolo Clínico – Meningite Bacteriana
  9       1.1672   Diretriz de Cefaleia Crônica Diária
  10      1.1914   Protocolo de Neuroimagem – Indicações de TC


##  PASSO 4 — Re-ranking com Cross-Encoder (Top-3 Finais)

### Bi-Encoder vs Cross-Encoder

| | Bi-Encoder | Cross-Encoder |
|---|---|---|
| **Input** | Query e Doc separados | `[CLS] Query [SEP] Doc` juntos |
| **Atenção cruzada** |  Não tem |  Bidirecional completa |
| **Velocidade** |  Muito rápida |  Lenta (par a par) |
| **Precisão** | ~80–90% recall | ~95–99% precision |
| **Uso** | Funil largo (Top-100) | Funil fino (Top-3) |

O Cross-Encoder vê as duas sequências juntas, permitindo que a atenção de cada token da query interaja diretamente com cada token do documento — produzindo um score de relevância muito mais preciso.

In [7]:
print("Carregando Cross-Encoder...")
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
print(f" Cross-Encoder '{CROSS_ENCODER_MODEL}' carregado.")

Carregando Cross-Encoder...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

 Cross-Encoder 'cross-encoder/ms-marco-MiniLM-L-6-v2' carregado.


In [15]:
# Criar pares (query original, documento)
# IMPORTANTE: usamos a QUERY ORIGINAL, não o documento hipotético!
pairs = [(USER_QUERY, doc["text"]) for doc in candidates]

#  Calcular scores de relevância
print(" Calculando scores do Cross-Encoder...")
ce_scores = cross_encoder.predict(pairs)

for doc, score in zip(candidates, ce_scores):
    doc["cross_encoder_score"] = float(score)

#  Ordenar por score do Cross-Encoder
reranked = sorted(candidates, key=lambda d: d["cross_encoder_score"], reverse=True)

#  Imprimir ranking completo
print(f"\n Ranking completo após Cross-Encoder (★ = selecionados para o LLM)")
print()
print(f"  {'':2} {'Rank':<5} {'CE Score':>10}  {'BI Score':>10}   Título")
for rank, doc in enumerate(reranked, 1):
    marker = "★" if rank <= TOP_K_RERANK else " "
    print(
        f"  {marker} {rank:<4} {doc['cross_encoder_score']:>10.4f}  "
        f"{doc['bi_encoder_score']:>10.4f}   {doc['title']}"
    )

 Calculando scores do Cross-Encoder...

 Ranking completo após Cross-Encoder (★ = selecionados para o LLM)

     Rank    CE Score    BI Score   Título
  ★ 1       -0.9746      1.1672   Diretriz de Cefaleia Crônica Diária
  ★ 2       -4.9316      0.7923   Manual de Oftalmologia – Sintomas Associados
  ★ 3       -6.7814      0.4550   Manual de Neurologia Clínica – Cap. 3
    4       -8.9419      1.1326   Neurologia – Cefaleia em Salvas (Cluster Headache)
    5       -9.0961      1.1914   Protocolo de Neuroimagem – Indicações de TC
    6       -9.5721      1.1512   Protocolo Clínico – Meningite Bacteriana
    7      -10.8874      1.1455   Tratado de Medicina Interna – Tireoidopatias
    8      -11.2283      1.1321   Farmacologia Clínica – Analgésicos
    9      -11.2485      0.9293   Manual de Neurologia – Cefaleia Tensional
    10     -11.2850      1.0782   Protocolo de Urgências Neurológicas


In [14]:
# Top-3 documentos finais
top_docs = reranked[:TOP_K_RERANK]

print(f"  DOCUMENTOS FINAIS INJETADOS NO CONTEXTO DO LLM (Top-{TOP_K_RERANK})")

for rank, doc in enumerate(top_docs, 1):
    print(f"\n  [{rank}] {doc['title']}")
    print(f"       Cross-Encoder Score : {doc['cross_encoder_score']:.4f}")
    print(f"       Bi-Encoder Score    : {doc['bi_encoder_score']:.4f}")
    print(f"       Texto:\n       {doc['text']}")

  DOCUMENTOS FINAIS INJETADOS NO CONTEXTO DO LLM (Top-3)

  [1] Diretriz de Cefaleia Crônica Diária
       Cross-Encoder Score : -0.9746
       Bi-Encoder Score    : 1.1672
       Texto:
       Cefaleia crônica diária é definida como dor de cabeça presente em mais de 15 dias por mês durante pelo menos 3 meses. Causas: enxaqueca crônica, cefaleia por uso excessivo de medicamentos (rebote analgésico), cefaleia tensional crônica e hemicrania contínua.

  [2] Manual de Oftalmologia – Sintomas Associados
       Cross-Encoder Score : -4.9316
       Bi-Encoder Score    : 0.7923
       Texto:
       Fotofobia, definida como hipersensibilidade dolorosa à luz, pode ser sintoma de meningite, cefaleia em salvas, uveíte anterior ou enxaqueca. A avaliação diferencial inclui inspeção do reflexo pupilar e avaliação de rigidez de nuca.

  [3] Manual de Neurologia Clínica – Cap. 3
       Cross-Encoder Score : -6.7814
       Bi-Encoder Score    : 0.4550
       Texto:
       Cefaleia pulsátil acompanhada 

## BÔNUS — Testar com Múltiplas Queries

Execute o pipeline completo para outras queries coloquiais.

In [13]:
def run_full_pipeline(query: str):
    """Executa o pipeline RAG completo para uma query."""

    print(f"  QUERY: '{query}'")

    # Passo 2: HyDE
    hyp_doc = generate_hypothetical_document(query)
    print(f"\n[HyDE] Doc. hipotético: '{hyp_doc[:100]}...'")

    # Passo 3: Bi-Encoder + HNSW
    q_vec = bi_encoder.encode([hyp_doc], convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(q_vec)
    dists, idxs = index.search(q_vec, TOP_K_RETRIEVE)

    cands = []
    for r, (d, i) in enumerate(zip(dists[0], idxs[0]), 1):
        doc = MEDICAL_CORPUS[i].copy()
        doc["bi_encoder_score"] = float(d)
        cands.append(doc)

    # Passo 4: Cross-Encoder
    pairs = [(query, doc["text"]) for doc in cands]
    scores = cross_encoder.predict(pairs)
    for doc, s in zip(cands, scores):
        doc["cross_encoder_score"] = float(s)

    reranked = sorted(cands, key=lambda d: d["cross_encoder_score"], reverse=True)

    print(f"\n  Top-{TOP_K_RERANK} documentos selecionados:")
    for rank, doc in enumerate(reranked[:TOP_K_RERANK], 1):
        print(f"  [{rank}] (CE={doc['cross_encoder_score']:.3f}) {doc['title']}")


#  Executar para outras queries
outras_queries = [
    "pressão alta dor na cabeça",
    "dor de cabeça todo dia ha meses",
    "tontura e vomito com ouvido tampado",
]

for q in outras_queries:
    run_full_pipeline(q)

  QUERY: 'pressão alta dor na cabeça'

[HyDE] Doc. hipotético: 'Crise hipertensiva com cefaleia occipital pulsátil, PA sistólica acima de 180 mmHg. Investigar lesão...'

  Top-3 documentos selecionados:
  [1] (CE=-2.889) Manual de Neurologia – Cefaleia Tensional
  [2] (CE=-7.348) Protocolo de Pós-Operatório Neurocirúrgico
  [3] (CE=-8.783) Neurologia – Cefaleia em Salvas (Cluster Headache)
  QUERY: 'dor de cabeça todo dia ha meses'

[HyDE] Doc. hipotético: 'Cefaleia crônica diária presente em mais de 15 dias por mês durante período superior a 3 meses. Inve...'

  Top-3 documentos selecionados:
  [1] (CE=6.474) Diretriz de Cefaleia Crônica Diária
  [2] (CE=-6.039) Protocolo de Neuroimagem – Indicações de TC
  [3] (CE=-7.065) Protocolo de Pós-Operatório Neurocirúrgico
  QUERY: 'tontura e vomito com ouvido tampado'

[HyDE] Doc. hipotético: 'Síndrome vestibular periférica com vertigem rotatória, náuseas, vômitos e plenitude auricular. Prová...'

  Top-3 documentos selecionados:
  [1] (CE=-

## Resumo do Pipeline

```
Usuário digita query coloquial
         │
         ▼
┌─────────────────────────────────────────┐
│  PASSO 2 — HyDE                         │
│  LLM "alucina" resposta técnica          │
│  → resolve a lacuna semântica           │
└─────────────────────────────────────────┘
         │  vetor do documento hipotético
         ▼
┌─────────────────────────────────────────┐
│  PASSO 3 — Bi-Encoder + HNSW           │
│  Busca O(log N) no grafo hierárquico    │
│  → Top-10 candidatos (funil largo)     │
└─────────────────────────────────────────┘
         │  10 documentos candidatos
         ▼
┌─────────────────────────────────────────┐
│  PASSO 4 — Cross-Encoder               │
│  Atenção bidirecional (query + doc)     │
│  → Top-3 documentos finais (funil fino)│
└─────────────────────────────────────────┘
         │
         ▼
   Contexto injetado no LLM gerador
   → Resposta final ao usuário
```

| Etapa | Técnica | Velocidade | Precisão |
|---|---|:---:|:---:|
| Transformação | HyDE | Rápida | Resolve lacuna semântica |
| Recuperação | Bi-Encoder + HNSW |  O(log N) | Alta recall |
| Refinamento | Cross-Encoder |  O(K) | Alta precision |